In [31]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated,Any
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator


In [32]:
load_dotenv()

False

In [33]:
model = ChatOllama(model="qwen2.5-coder:7b")

In [34]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(..., description="Detailed feedback on the generated essay.")
    score: int = Field(..., ge=0, le=10, description="A score from 1 to 10 evaluating the quality of the essay.")
    


In [35]:
structured_model=model.with_structured_output(EvaluationSchema)
essay ="""This is a sample essay that needs to be evaluated for its content, structure, and overall quality.
Cows are gentle and useful animals that play an important role in agriculture and daily life. They provide milk, which is used to make butter, cheese, yogurt, and many other dairy products. In many rural areas, cows also help farmers by supplying manure for crops and, in some places, by pulling carts or plows. Cows are calm creatures that often live in herds and need proper care, food, and shelter. Because of their many benefits, cows are valued in many cultures around the world.
"""

prompt = "Evaluate the following essay and provide feedback and a score from 1 to 10:\n\n {essay}"
evaluation = structured_model.invoke(prompt)


In [36]:
print(f"Feedback: {evaluation.feedback}")
print(f"Score: {evaluation.score}")

Feedback: Your essay is well-structured with clear ideas and proper grammar. However, it could benefit from more specific details and examples to strengthen your arguments. Also, ensure that all sources are properly cited if you have included any.
Score: 7


In [37]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    ananlysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    average_score: float


In [38]:
def evaluate_language(state: UPSCState) :
    prompt = f"Evaluate the language quality used in the following essay and provide feedback score from 1 to 10:\n\n {state['essay']}"
    evaluation = structured_model.invoke(prompt)

    return{"language_feedback": evaluation.feedback, "individual_scores": [evaluation.score]}
    



In [39]:
def evaluate_analysis(state: UPSCState) :
    prompt = f"Evaluate the depth of the analysis quality used in the following essay and provide feedback score from 1 to 10:\n\n {state['essay']}"
    evaluation = structured_model.invoke(prompt)

    return{"analysis_feedback": evaluation.feedback, "individual_scores": [evaluation.score]}
    



In [40]:
def evaluate_clarity(state: UPSCState):
    prompt = f"Evaluate the clarity of the writing in the following essay and provide feedback score from 1 to 10:\n\n {state['essay']}"
    evaluation = structured_model.invoke(prompt)

    return{"clarity_feedback": evaluation.feedback, "individual_scores": [evaluation.score]}
    



In [41]:
def finalize_evaluation(state: UPSCState):
    prompt=f"Based on the following feedback and scores, provide an overall evaluation of the essay:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['ananlysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}\nIndividual Scores: {state['individual_scores']}\nAverage Score: {state['average_score']}"
    overall_feedback = model.invoke(prompt).content

    avg_score=sum(state['individual_scores'])/len(state['individual_scores'])

    return{"overall_feedback": overall_feedback, "average_score": avg_score}
    



In [42]:
graph = StateGraph(UPSCState)


In [43]:
graph.add_node("evaluate_language", evaluate_language)
graph.add_node("evaluate_analysis", evaluate_analysis)
graph.add_node("evaluate_clarity", evaluate_clarity)
graph.add_node("finalize_evaluation", finalize_evaluation)

graph.add_edge(START, "evaluate_language")
graph.add_edge(START, "evaluate_analysis")
graph.add_edge(START, "evaluate_clarity")

graph.add_edge("evaluate_language", "finalize_evaluation")
graph.add_edge("evaluate_analysis", "finalize_evaluation")
graph.add_edge("evaluate_clarity", "finalize_evaluation")

graph.add_edge("finalize_evaluation", END)

workflow = graph.compile()

In [ ]:
essay="""Deforestation

Deforestation is the large-scale cutting down of forests and trees by humans for various purposes such as agriculture, urbanization, industrialization, and logging. It is one of the most serious environmental problems facing the world today. Forests are essential for maintaining ecological balance, supporting biodiversity, and providing oxygen. When forests are destroyed, the environment, wildlife, and human life are all negatively affected.

One major cause of deforestation is the expansion of agricultural land. Farmers often clear forests to grow crops or raise livestock. Another important cause is urbanization, where forests are cut down to build roads, houses, factories, and cities. Logging for timber, paper, and fuel also contributes heavily to forest destruction. In many developing countries, people depend on wood for cooking and heating, which further increases tree cutting.

Deforestation has many harmful effects on the environment. Trees absorb carbon dioxide and release oxygen. When forests are destroyed, carbon dioxide levels increase, contributing to global warming and climate change. Forest destruction also causes soil erosion because tree roots help hold the soil together. Without trees, fertile soil is washed away by rain, making the land less productive. Furthermore, many animals lose their natural habitats, leading to a decline in biodiversity and even extinction of some species.

Deforestation also affects human life. Forests help regulate rainfall and maintain the water cycle. Their destruction can lead to droughts, floods, and irregular weather patterns. Indigenous communities who depend on forests for food, shelter, and livelihood are also severely impacted.

To reduce deforestation, governments and individuals must take action. Afforestation and reforestation programs should be encouraged to plant more trees. Strict laws should be implemented against illegal logging. People should also promote sustainable farming and use forest resources responsibly. Public awareness about environmental conservation is equally important.

In conclusion, deforestation is a major threat to the planet and all living organisms. Protecting forests is essential for maintaining ecological balance and ensuring a healthy future for coming generations. Every individual has a responsibility to help conserve forests and protect nature.
"""
initial_state = UPSCState(
    essay=essay,
)

result = workflow.invoke(initial_state)
print(result)